# 🧬 Forensic Suspect Face Composite Generator: ControlNet Training
### Phase 2: Resume Training from Checkpoint-2500 ➔ Final Step 3000

This self-contained notebook resumes training your **ControlNet** on top of `SG161222/Realistic_Vision_V5.1_noVAE` (SD 1.5 photorealistic backbone) for Indian suspect face composite generation.

---
### 🎯 What this Notebook Does:
1. **Mounts Google Drive** and validates your saved **`checkpoint-2500`** in `IVP/checkpoints/controlnet_indian_faces_512/`.
2. **Installs required libraries** (`diffusers`, `accelerate`, `transformers`, `bitsandbytes` 8-bit AdamW).
3. **Deploys training & inference scripts** (`train_controlnet.py`, `infer_controlnet.py`, `dataset_manifest.jsonl`) automatically.
4. **Extracts datasets to local NVMe** (`/content/faces_512`, `/content/sketches_augmented`) for high-speed I/O.
5. **Resumes training from Step 2,500 ➔ 3,000** (Only 500 steps remaining, ~8–10 mins on Tesla T4).
6. **Saves `final_controlnet`** and runs **interactive visual inference** with side-by-side sketch-to-photo comparison.

> **GPU Target:** NVIDIA Tesla T4 (15–16 GB VRAM) on Google Colab Free Tier.
> **Zero Data Loss:** All weights are saved directly to your Google Drive.

## 1. Mount Google Drive & Verify Checkpoint-2500

In [ ]:
# Mount Google Drive and verify persistent directory
from google.colab import drive
import os, glob, torch

drive.mount('/content/drive', force_remount=False)

DRIVE_DIR = '/content/drive/MyDrive/IVP'
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints', 'controlnet_indian_faces_512')

assert torch.cuda.is_available(), 'CUDA GPU not detected! Go to Runtime -> Change runtime type -> T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f'[OK] GPU Detected: {gpu_name} ({vram:.1f} GB VRAM)')
print(f'[OK] Drive Directory: {DRIVE_DIR}')

# Check for existing checkpoints
if os.path.exists(CHECKPOINT_DIR):
    ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint-*')))
    print(f'[OK] Found {len(ckpts)} checkpoint(s) in Drive:')
    for c in ckpts:
        print(f'     - {os.path.basename(c)}')
else:
    print(f'[WARN] Directory {CHECKPOINT_DIR} not found yet. It will be created.')

## 2. Install Required Dependencies

In [ ]:
# Install Diffusers, Accelerate, Transformers, BitsAndBytes (8-bit AdamW)
!pip install -q "diffusers>=0.35.0" "accelerate>=0.28.0" "transformers>=4.38.0" "bitsandbytes>=0.42.0" torchvision safetensors scipy

## 3. Deploy Training & Inference Scripts
> Self-contained auto-deployment: writes `train_controlnet.py`, `infer_controlnet.py`, and `dataset_manifest.jsonl` to `/content/` and persists backup copies in Google Drive.

In [ ]:
import gzip, base64, os, shutil

# Embedded compressed payloads
TRAIN_B64 = "H4sIAMQL52oC/9U8/XPbuHK/669A+abvqFeJlpzkmnMf34zjjzzPix2P7eSudTUcWoRsnimSJUh/JHX/9u4uABIgKclOcjNtZmJZBLBYLPZ7l3YcZ1AWYZwG8ywtiyxJeenlj2zMDrOCpyKes/NbXs5vxmU2PgznnO3JeSe8ZBe4ME6v2Wmc8yRO+YCe8IhlKXsz3X6A/+wojeIwZeeVyPm8ZMsqKePxMovChEVhGQqAUwkEsh8vFpXghWB/ZrvzOU94EZbcG5xXeZ4VpWDTB5YVbPuBnXw+2j/aZWcXv7FXk18m7P3pJ8Hc/f3TIQvTiO2dfmJR8TguqpTd8SJexPOwjLPUGzhw2kG8RHAsE/q330WW6t+/xPkiTrj+GhbXeVgIPlgU2ZLlYXmTxFdMDZ7CVzlwevRBPzxahte83qTMivmN9cVLU29RpXNECEgQCnYoYcjRqowT4SFhNMB9SaQR/fIhCyNeGAvuYgGA9Fy4yVQssmIpBnJOWNNRT6kpmxXtKV6SXV/jTaip17wM8BHvziQ09TzALhCcRwotjQPepJqx9+Ho9II/lMdZxJOR/Jrd8jT+omFH9eWrJe6Awb/dqsx4OodlxT8+jOhRw38SGj2Euz8+n9/wqAIE5aNPMGN7H2ZHMdLamHxehlcJl+wGIw3Amo2HLaS8LC/jZfyF2MgkzzwD1uWBUFsH93F5E9yHxbLKBwNJO+YbhHSDIA2XPAiGg8Eg4gsWwgEDoEwRzstAyYNwr+AjiOJih3hsuENoA/MiOcZqtqjFB3hWADewRShKlmRzlKxY3LJ4wUpgYJixyJIIaRtlLM1Kxh9iUXooDAj3T2zvhs9vWZ4JEQNhmMiqAgQdwBI0PDOIlwQMAriXJeEVe59l1zB3v4jv+JDgCB4CQyLaAs58qc8wojO4zhYqGJ6WWxEu2Tp+pKVbR59PnWF7jjOcEUiF34U8hTo4XgHABWWSFY/MzQu+ACrrpXDWQlLi5PMxcH0q8R2xrLzhxX0sONOYSbQliYIiA8L4HTyQiO1nHtFPuFLbIEHrC2NbzFmAlhQBqD5zJk+MjQfqXFOPoUoVqCvpkVyKcHwLLwuqMTNMEOf+zWkaYI/41XC7qNeA6iHJbbQJkDJCEhpX24zqGQvkaJx16ShdFUymwaFGxAM+ckbMkXaAntePZzY0gvgFThTBUQhsZxgOtPjSg6r5Ly/itHQXzuXBbxdnu3sXM3agOAd029fFl6daWEhMvtbUefI8zxn2wmwouLyFny5YBeAF4V8UFR9JgQqyW/raDwA1g7Yu3n/E+SF8uosvQJgCuAzMwJf+w+A/PDAdIEwSt8ZkuObwzuXHf8wUcxliM8+WecJBf6845VXBw9tBm94d5uliKhcqtt72lM8Am+9W10sgFBgHUhHqcT+L16OhXuTYq7rsbjKV3jPYtZcrGTD33iDBPYj0CL2J1g8Vnlc9R+mTohWzeoRKbBQq8X1CJTpCZZJ7jVxZt/IjRUt8g2iZyGyWrprHv1fAejlzpYwN5kkoRO2eK65x1WfjK6CrKGov/LrIqjQal0UFtCKJJj6ea/cIb1HjAX4umNeI3cUhW4ZpvODKWyDQ6LUEAfj9ZRC4gieLUT0pQAd5h4kSjH6p/bsRE3kSl/QYuNChaANYGbcOoiLLswoGF0kWonBPvMkb4+wI36tFU5lna7uhJ/nFK7jIkjvuNoTv960skMOBvVWNNWqn2kO1pph4wyzzqz0RkCpiTr7QrNkGvC0gfByhS06KT6oHii2EDUApDpxFUCyOsI4xsoba/sq60T5vrJ5uoa2csBjjm8bJZ+O/scvxdMSmMxv5eHkdNOEIkrP+4u2BlGSCu/aJjAlnXADhXRc4d4QsDA4iyB0vcnDlkFt9Y+6ROYBevvfu6MPRycHuWevgxpqL7AKEJyvc1VNO4GeYIBaXE+/NbMTow6DN0LrTrhy1aDTpkkjO+/9ApRWnPlaCqHi4Ebx5WYVJoOW0X24HhgZEc9patEIRos0ksSHD2hKSrs6UHoNyMKz9+4xgvyG34HVOBmvWaPqBZaOynKduCwRZKTBzGOQC8/hOVS7Gb6XdWnTPjuEpnn2x03cApCPO8EDXxvmqQ6Dox2nV9QMo7eBTPsRL0Hy4CGzY6y+gTmc+6HMwnA6GhbjYAw3hOjQGJ4BRqfr7Lb+hI70wB9JELsIwOEx7bsZMGxaYEgirPodJxQ+KAjh14ZxkTIRohIF8aPKQWF9bNH8iUsoj/PSVPp9+coamfUt4qsybQcSCl1WRMhhzTZzMhcoKEYsp61jwpDGMQ1QGKA22o/hMls4bZtZQ285EvoaDFf65NaDDf2BkXqRolmS2gIGrCAzZht/nHCPm9SnXy6CeBuY6T8ANcfvgbfRzneG3ia8igCmz6pHtaBjkNXgCmDsu+bJ2e+LowdgHh+CcJmNcwgzLhkLAr8wopVJlrrDWvDIawl01HJudcIdLR00jO+zMLDVqQFhBiHrxNWxB23uklIyVQ/Rw7ngBgnz2/p1BaUxh7LRY5xCk/yrE5BFe5o1kHGToZSyETCWiuxBVBX5RSVmxCaWU38vNR8w0bvMsyQrfnW6/BVuqfgwb/PL4AS7tDpWB0AS0vRC32cuyYxCwWgZcMtzAjlE2XozpSwdyTet+DEAr7kfN6NyPsfIF99MDbT1pt9+8GTH9wyCtdbQ+OndcGbfZ2yL1K4+dFuDYlDbFyTzuHb5n+y1POpeTfRIvaV7kI1QSjk3djnf+NzahKEem1gG3yJ0OPYQEYfxfuwtaylZvrSMf2ifNK2CgqD55HSW4PYtt12oZPqBduS5vfHupt8TkdNCM2+vyMJKuQTPBaTttVSoTtTJqHnSVXlCSLyd8B2hXjwM19IEuJ4aqUlrxqwXIMW/e2bEEzt7SWckwsGzlWAtGjRmsqX+vpzypBDqVZwKQ61qW6AkGcLp44+0W1xUajlMacSMu5uAeEbkcqliZRS3UXqtqX67MfV+8Zn9m++BRUPlpqNhQZb8oRx/xEotdZpKcAM/R1OYZuOnyJHOEF1AM1kk+d0OzLWO52hRIEFZJGQD35hSPgo/hGlB7Etg23E6S2vHMbbaasiDE/GSTjRSzQW8PmBSvgQjtOuNxDjwkq4GBZG8qfGQFKTGQ3vIx5z6lCtQZfOf8/fTn6fb29tYZh9BLlPE8+Ez1reDzG28apNnn3YMW55v/bniS+87fKypi0X3RxuxoH/1T0rq1wdphIDL34eMy2RJUDhpHuh40vpuO3yjyrjibFUn0n0XlG5p4Br3qZCP2eFWYSVuxfC1WkgfQgelFyWaVTYh8lAzVFFrIZQ7veGQyMXMVVCpAST4HhstU1WW4HmFZeL4K0VBgKKvRBsgN2q83YfoO1zNcz3KQ+/enn5j7msWCUdEOhHRRF61AcKc/v383Ym/pYV1A3n79/t0GXK+LMIrh9yCcz6tlJQPpQJQ8F9+G9nsFkJkAGQFkLl8sgOxIzavmdD4gj6Sd4hlH7NU2ftmm6vcG5BOIKEjdYvFWo0sptwbhKR+/2YSzoSY1SEbFZc0GOwzBDDdJz0Mgr341+V5NJpNN2FxkJdxuqZsQFOn+ZzoZT7cZz7P5jaBrfj16O5nIeTo23IBhw+Hky63E8s1mJM9BZAyJYRxct0f2m8IWhMayEVtG/XY9iuCAwu8B1qmDBny/LgLeAhXybN1jYAvU+0mu/glHMKOqdmZUIb+Jr28wD2IsASIbimg9I4DxjwIwFHPS8v3IL/Lpz5guvslisDv+pZNm6Pupx87VAj5na7fB3oR+Gd1euw7ClQCsBCyV2X3fEaAJOfBuxTcS86xC2fyvKobgSN41KsXTT1uon4CU1JjyCD6UbDmgzN11EZePimQ6VpeomV6OcnyWwM7a5cEBUA/2rDrkPErB3aJcptn9QaJxTN04iNL+/qkEZczwzfmNf7tGFfq4ubdmQkO21uXLla2HFNSS8z6votCLRRDehXGC9rp2V5AdzHDhdz4n1pMAG1akObpTQTasuDQFf1PkwiRCc2LcD6kM+GTAekYuJgNMw1uOJT23tc3KilVdQDs6ObqYGZcSWbfy32yf3wGn72DWqkEloodPMHxSLTGEQoS4aM1Kq6VGlosnZ2hWZCHiGEfZfYqJPVSXdQMO+LLUHCM9UumrxZH0n4W31o17EdFqAux//PXkw8fd/Rn7oHCRfhpqFDiQxuCJuaRvNNJCqpy/H+q84DzEtpuhPmZZPDabmYUcq93II43ZnMrV242YqK5k6st36uVGiFnyhzJQzUgaqu5peiZUA4IB+C5E2261Oj0PHiw0wFTgowOcvr6n54FDAAa8NItF09qER7a6rJ4Hs16uAPOHOYeg+oA+UMJDwYysBSYYdRRAaUUNtB2aK1b6dffsZAbiAH4eWFXJRgYDcUwKA0vxJ+n5I69RugoU8Hr/v1uxNgTD2RA72LH5j+fEP5QbfyBH/hFc+YdxZn8KTVYYuFakr9CawqWDY4huL8YQ0huiNALygpFKUHac/DQ0fljerjUm6tZ+H45SVmBhzDRhy8gMLYlZBwsrNMr9sw8GE2RVOWo6VShP0N7Ku06yK9dpYI7/ItviIlT12LAxnLWLBAS8m4L/EzvHxkmwUuD8gJubPLKrR/KOOlM1egIWwA3S1xG75Y9+Ei6vopA9YFKodB88tEce1XDAZ3OGl+PpjNDrHQGUo/gaHijvYdItJEhqBbgjlvdxY1zZU0wxLxZxMVb2YtVbs3iO5exVfWcH55+OD2bskKpcxCeo3pp7Ar3XxgjUIPErTiRjSth/NY6yolWnyQKhrrE7cDvihikoY+ehDa4rY99HCu1Q7SUylYyahmnFeAXh1K3A28lYyu8N1L/1nKiOXPxhag1MeSiBRQYr3DUi2cr4v/TgnfvXPtQ9h1isVE4S9trHi9hK1qCvuBqtnot/0aWvO7FdlAAuA+exqNsC1q3sk6PBeims4ZMWqL91JH+N0v+2O/lhjPhiJrRP8BLsfyDmL8JamdPX4JYVnH+RrdHSxRLMxf1H7PPuwYihT8MOJCpDNoYgGoxG2crZD5TPAlwEATewRIARaOCCyydUBGY6O2umIXZrho13ZFqTmljPmEN4NqH4gXQQdHTMrCyTzJnW6bUyo2Qr+3y2e6x9hptQhCWl9/UOI+ZwAhrUIbcF1DGUjYHX2jUNvm889lG+98DBiX47vopLthuFy1+lx4D1D51BBc8KnKIoLCI9Q+XZCZSVftSBpfWQpmV6s2CeUC8URf701COo3TBvXX6gJdPypQ04gwBErx5L7IAU7Cq9sqa1cYBxE4O3sP6HqIqPpxdHxzP2iYrk7wCp3TR6R0iZdK6x8VaET98n9QoJa3tqxdJE/Df1atbp4wVSmaUh5aQJOU/H3TWSQC6LfK75SllYgCYveSHc4YglhW9d/4hd8TIUvjvxfhkx+PHLLzBLWrQg4vPwEdPT2zVn/uzp16KwENe8F6Ua2DCzLPuTdLtp3WSGnKfyeip/hP1LxkL9Uo3f6bC1ij4qXWU+Mnpe/Xb3q29gNTR2a2ofmN2vnXkLQ3rSnt7COKHzYyBUE6PJ11kHa3JlDTC/Db2ZJG6qxSLhrbIy5pnus+IWrtOfdLAG5QDeEDW2Y/iRlo48CLid7vYIx+Z5FczBb0VrjK/0GT2IOSCy5MusePRXSrZK5Slm+FePfdDFiDNUMHUgONBVd5mA9dn2Ggq3qhJScRVWlLnhja+G4rUY+PVvLeLRApUzXXHvr8ziAi7SlQ61rD6YTY632GHBsSladlWYL/zZdmDU4Dnq8NKofXhTu+RyA7fHuLwAqI33YSjKMRiWezQi1t5AncKKWsJUNroueURheGO/tKk03cK/sYndbBdQv1qYXnPXmNeyFyYGHo67w+/QtWa8ZpwT/PMPZ6w5KVr9bkTG9qoCW9xhLsZ0JmL0ZiEADBIIwy8nsx1vm9d5X60/sfBRm1Oq+r3a7svctpPvKLxUZmlOtBrk9OdBHQttAko1mw1ArzTU2rkrM7ebER8xWuabMHo8vheuJTfwBWsUC//iNa9Ef8iy/JvS4/+ZXl6c7R6dzPA91YJcQy34O+w4fIDHKP1fa/HHosAxUhnlXhVTvq65gie7bcpUPP1uwsXBOQQH+2f/Pj77dMKOP+4f7LDPVL5C5LY75S0ZFxAozB2FiQ7PDK5eIaiU+3pZNaFProhWok0+M+VhIPa0ZdCyiWlQawVxSboPP9y2MtNSdhOjU28c9K+NzTEKEqbbWptfgJ0Cn7r1ZkZWQbp652WWH5VIDcvdeyaGz9nQ7lE3iV+X77jhyg3bPaC7ngrQ9NvG1POpIkZcDJpL5OHc7jdvdREScpd2h9kMRXCViJrpupRShagjpLy7JpihJ6cEEb70LOv/7ioY+re/EDQ49CK+9sQ8xMoBNj6hEW0d/53HzgmoTEvLhsN4yRvTbKWtawWH7YhpkMS33FW7thL8Goa1Apm+E1RA+NVKiWvUa7chqMGNmN7PEzdhzsFqjIZYE0f15ush+dXaCQiZNeGheajHoKFgGxEsqtMzvavCddQccNgm6V7DURj+y15KO4ko9XpwE0cRR48N041IKEPpu4qnmj7C2dBqcpSb7Xut7j9yQHJ8x44EQr3NY3YttrM17bbfhp9X90E+i7mx9BlQAgYbjgPVvDICTzrqPJbvoikh7fKIdU/d5oWGPTpDvbT2e5+O1iRU8Q+MRH4PtUYr3lMAgZ2XPmVfbD5s3+CBJzNW3bszLhYIFWPKQvQU9XJ829Enq/9/hnDG1WP7r/xzIUF9Cv9S3voKLlLdesQX4Oz289Gsu2vDWD2b+n1stwKBF19qj2QeQmCXwVXiS7VVKfvj8M89IAilTLcYUD4GvTTseadAayNb1yNECO69Jagp/OI2PCA9WUxQSADNdxisVA/Qkoep0+JB02hi1gvZ0EXgw3XJIvGYzuskXN/La8bceRLnMtWYZsUyWJ1amXqtmlaTQ2pFMZuiHHsx/M8IAdd+Qeu5BzIdpH/x2fS7smgJujxrA6GVdRP0ntnlSv9vBs40Mh0GWvChXhbY8V4v0M1WAZgVZbUKnJi0XRGcir6ql0mXf5bOeE/3IYZNE0yVWA613+doWrWWUL2igqk5VaD6HQB3m5YWZoHXos6KN9mr9L7At/aiVmpAPpe9QqbXuB6IJzE1aoga8+H6Kuj57mfw/c+pL9nQ9kYFHJzQrzU0oydqOvHkbR3G+IeYjMVUsmg3w3n3YVwGoFYD6uDMUu1EPpd3F7iNeo19/V04cmpDPKvJ5xvJvprUNWLDvlh0/+MJELgOa/UfOPgnRTbZciP7wssMpKOGRqQexPjinPyTRxT6B5I6gQr/ZRPj4H8BJMGudIZMAAA="
INFER_B64 = "H4sIAMQL52oC/+1Y32/bNhB+119B6KUSZiu2u7RdAD20cWtkaBKjLrKHohAY6SQTlUiVpJJmRf/3HUnJlhzH7va4LYBRiTwev/vup+r7vsd4DjJJBddSlBx0VD+QMXknJHDFUrL6Ajpdj7UYv6MpkAsjDTwFbwEcJNWgSL0WGsVpyZTGE6fT2Tf8kdzIp6KqhWIaSC5FRSgnjNeNJsqqxfeMaPimSY27tY68VVPXQmpFRK2Z4LQkpbgfS1CibMwCOT//eHNSiVtWgrsYr+AZM3uMFyRgVTHDH6lEBmHk+Wihxyqjk1BZ1FQq8CyUmup1yW5Ju7nEV7exvHjfLV5UtIDuONqYtiIZy/NGgVSdYOAR/Dt3JF6BvsTby5FdnC8vV6K8A3nZlBoZgnqVriFrSpBOYKXpbQlzqxKt2CpZshpKxuGI2AVajL9O2gs9L4OcWFMTtFkF4ZnVYFckiTdERK9l0VTA9dLuBBmoVDJLfOx3/u2FQqNqSDWxgWB5cMHhhz31Ec0yc6vVG/jj8TayEkO5PyL6oYZYaTkiiJMiK7EfnSAn6ZdaMK7VSe8IQ99SnphYUglG1UnOMCh64eo7dvb9raGsY984Fn1HtKTITtZzEtneediEW4pMmoAq96NfLaYvprPZ7ORDlwTJDTNOSm5Oo2nCxc3rt0dxvsFLyGpOptHpbkbZq8nF/DBKl1KPSZbwtWESsvijbOBn2RpkKTNpQIIusZdXi5Pfl4vwMByX0f8cySbs7pnmoFS/Thy+mUNBNbuD5DGEJ+/c+PK2bKR8GJmyQ7426AGNLymVWgg+Is8zNIJnYLyPvsHUh8w+5qxopH2W9B4L0ajl7qjbr1qwrWUkF5JsPS8ajX44bC7jDPPEuGhvdF4JfpTr67bYnu2ttq7MDivrkVDUyFKxjcO8FFRvIU2iF6fHIM2BC8whLOmdMnK/Bk4au9a3mgSTaGJidhpNTIxhy6mMjzKC90zGk+jl6RG4juUD5aloa2GWKFcDo5oXR117bdW22aOo8bG54WerZb+vJSqlJTxJ56ujdP4BrFhrIvIup4dd0zA43kfgK+TvtyP08aZKWDcXJKbDqQ4pltYtztnfcjpqIcFsMn4+IUzhMKG0nRbMXFDR8gikomEZtWgO8PYyOgrovKRKsZyBHOcSgHR6idN7OAkAsr08/Dpz5yToRvLueL9fuw5eYcPqerdZx849EHJ1644hnJj4aZNRn7DczSmReY2YSugdZaWZHIKQQIktxk/rxrdn721MJJlBiBrcOcvR9IVR1OneKLfne2LPZ54jQKJ9Qe5/en/9ev6ZvBc0Mz7sdVozLZyR7wZ4tDMQ/GhZ3C4jlp1JKjLnsZxD28SDfYpGDpqzJ+4bFzqYbeTHbqyLRA3cKep1ztBoxWENPfhh8cYPIyyG7E8ITPMbmdE2dGjbiiDkhrhFtxI43mL3TxhVlGMfSUw4tNfhUwsJWbZLvWqGwc6FJqZs24A3HTnYEQoj+IYNYjPZDZxweT1/+5m0U6Hpnmv0yP34A07qS1vJz3upf0aefd9R/uMZCbqaG7vN7vVHl3YWuztR7CG0j/Q4n1sTmA3En510H0XFIJktkO3kNsz0bejE28ehyFPBNJRSNAf9kNhJEqRttpv9cGBYpLqxH0088FHgrEJUOFIMLRpqiZzIEA5+lCRfqJRUJYoVFVV7Ri1aFkJiVFSJtc3P6kpZLL/84j+FXYs2qnvecuEaW4EdoHaUia0H3PMQwc6A5gR3Focn7F1xF3F7feniLXa5vOOkLpoHwbyD6XETa3E93hieHPYad2i49lTs7Wnx8W5leyyyc3tXduLN09aJkeVEfZp8tmumfu8WjK5efMTperxA2Nj23WfdoE6QwH0045BLTI+GfiU4lrf/J+x/NGH3JeO/PdG8jfmRmfldQ+x9YYTDkWl+fYXZt0LJjGw+M3r/baZFNzf1dJiZycPZIUk4rSBJ7IyWJGZgTBLfpbibHr2/AAmW9JHjEwAA"
MANIFEST_B64 = "..."

# 1. Deploy train_controlnet.py
with open('/content/train_controlnet.py', 'wb') as f:
    f.write(gzip.decompress(base64.b64decode(TRAIN_B64)))
print(f'[DEPLOY] /content/train_controlnet.py ({os.path.getsize("/content/train_controlnet.py")} bytes)')

# 2. Deploy infer_controlnet.py
with open('/content/infer_controlnet.py', 'wb') as f:
    f.write(gzip.decompress(base64.b64decode(INFER_B64)))
print(f'[DEPLOY] /content/infer_controlnet.py ({os.path.getsize("/content/infer_controlnet.py")} bytes)')

# 3. Deploy dataset_manifest.jsonl
drive_manifest = os.path.join(DRIVE_DIR, 'dataset_manifest.jsonl')
if os.path.exists(drive_manifest) and os.path.getsize(drive_manifest) > 1000:
    shutil.copy(drive_manifest, '/content/dataset_manifest.jsonl')
    print(f'[LOAD] /content/dataset_manifest.jsonl copied from Drive ({os.path.getsize("/content/dataset_manifest.jsonl")} bytes)')
else:
    with open('/content/dataset_manifest.jsonl', 'wb') as f:
        f.write(gzip.decompress(base64.b64decode(MANIFEST_B64)))
    shutil.copy('/content/dataset_manifest.jsonl', drive_manifest)
    print(f'[DEPLOY] /content/dataset_manifest.jsonl extracted and backed up to Drive ({os.path.getsize("/content/dataset_manifest.jsonl")} bytes)')

# Backup python scripts to Drive
shutil.copy('/content/train_controlnet.py', os.path.join(DRIVE_DIR, 'train_controlnet.py'))
shutil.copy('/content/infer_controlnet.py', os.path.join(DRIVE_DIR, 'infer_controlnet.py'))
print('[BACKUP] Training and inference scripts backed up to Google Drive.')

## 4. Extract Datasets to Fast Local NVMe Disk (`/content`)
> Unpacks the 2,000 photos and 6,000 augmented sketches onto local NVMe for maximum training speed.

In [ ]:
import os, zipfile, glob

# 1. Extract 512x512 Face Images
if not os.path.exists('/content/faces_512') or len(os.listdir('/content/faces_512')) < 100:
    face_zips = glob.glob(os.path.join(DRIVE_DIR, '*Faces_512*.zip')) + glob.glob(os.path.join(DRIVE_DIR, '*Face_512*.zip'))
    if face_zips:
        print(f'[EXTRACT] Extracting {os.path.basename(face_zips[0])} to /content/faces_512...')
        os.makedirs('/content/faces_512', exist_ok=True)
        with zipfile.ZipFile(face_zips[0], 'r') as z:
            z.extractall('/content/faces_512')
        print(f'[OK] Faces extracted: {len(os.listdir("/content/faces_512"))} images.')
else:
    print(f'[ALREADY READY] /content/faces_512 has {len(os.listdir("/content/faces_512"))} images.')

# 2. Extract Augmented Sketches (Original, Rough Wobble, Faded Pencil)
if not os.path.exists('/content/sketches_augmented') or len(os.listdir('/content/sketches_augmented')) < 100:
    sketch_zips = glob.glob(os.path.join(DRIVE_DIR, '*Sketches_Augmented*.zip')) + glob.glob(os.path.join(DRIVE_DIR, '*sketches*.zip'))
    if sketch_zips:
        print(f'[EXTRACT] Extracting {os.path.basename(sketch_zips[0])} to /content/sketches_augmented...')
        os.makedirs('/content/sketches_augmented', exist_ok=True)
        with zipfile.ZipFile(sketch_zips[0], 'r') as z:
            z.extractall('/content/sketches_augmented')
        print(f'[OK] Sketches extracted: {len(os.listdir("/content/sketches_augmented"))} images.')
else:
    print(f'[ALREADY READY] /content/sketches_augmented has {len(os.listdir("/content/sketches_augmented"))} images.')

## 5. Resume ControlNet Fine-Tuning (Step 2,500 ➔ 3,000)
The training script detects `--resume_from_checkpoint="latest"`, finds `checkpoint-2500` in your Google Drive, fast-forwards the cosine learning rate scheduler to step 2,500, and trains through step 3,000.

Remaining training:
- **Step 2501 ➔ 3000:** Only 500 steps (~8 to 10 minutes).
- **Step 3000:** Saves `checkpoint-3000`
- **Completion:** Packages and saves `final_controlnet` to your Google Drive!

In [ ]:
# Launch training with automatic checkpoint resumption
!python /content/train_controlnet.py \
    --pretrained_model_name_or_path="SG161222/Realistic_Vision_V5.1_noVAE" \
    --manifest_path="/content/dataset_manifest.jsonl" \
    --output_dir="/content/drive/MyDrive/IVP/checkpoints/controlnet_indian_faces_512" \
    --train_batch_size=4 \
    --gradient_accumulation_steps=4 \
    --learning_rate=1e-5 \
    --max_train_steps=3000 \
    --checkpointing_steps=500 \
    --mixed_precision="fp16" \
    --resume_from_checkpoint="latest"

## 6. Inference: Generate Photorealistic Suspect Composite
Test the trained ControlNet model on a held-out test sketch with forensic witness description prompts.

In [ ]:
import os, glob
import matplotlib.pyplot as plt
from PIL import Image

CKPT_DIR = '/content/drive/MyDrive/IVP/checkpoints/controlnet_indian_faces_512'
final_model = os.path.join(CKPT_DIR, 'final_controlnet')

# If final_controlnet is not yet completed, use latest available checkpoint (e.g. checkpoint-2000)
if not os.path.exists(final_model):
    ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')), key=lambda x: int(x.split('-')[-1]) if x.split('-')[-1].isdigit() else 0)
    if ckpts:
        final_model = ckpts[-1]

print(f'[INFERENCE] Using model checkpoint: {final_model}')

# Pick a test sketch (held-out test identity from test split)
test_sketch = '/content/sketches_augmented/indian_0001_orig.jpg'
output_composite = '/content/drive/MyDrive/IVP/generated_suspect_composite.png'

# Run inference
!python /content/infer_controlnet.py \
    --controlnet_path="{final_model}" \
    --sketch_path="{test_sketch}" \
    --prompt="high quality forensic photograph of an Indian male suspect, wheatish skin tone, defined jawline, neutral gaze, detailed photorealistic facial texture, 512x512" \
    --negative_prompt="blurry, cartoon, 3d render, distorted, drawing, sketch, low quality, oversaturated" \
    --num_inference_steps=25 \
    --guidance_scale=7.5 \
    --controlnet_conditioning_scale=0.85 \
    --output_path="{output_composite}"

# Display side-by-side comparison
if os.path.exists(output_composite):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    axes[0].imshow(Image.open(test_sketch))
    axes[0].set_title('Input Witness Sketch (512x512)', fontsize=12, pad=10)
    axes[0].axis('off')

    axes[1].imshow(Image.open(output_composite))
    axes[1].set_title('Generated Forensic Suspect Composite', fontsize=12, pad=10)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print(f'[SAVED] Composite saved to Google Drive: {output_composite}')

## 7. Optional Multi-Modal Prior: CCTV / Low-Quality Photo Conditioning
> When investigators have a blurry CCTV capture or poor mobile phone snapshot, pass it as `--init_image` (with denoising strength `0.65`). The sketch enforces structural facial geometry, while the CCTV photo conditions skin undertones and lighting.

In [ ]:
# Optional: Img2Img inference combining a blurry CCTV photo + forensic sketch
# Provide path to a blurry reference photo if available:
cctv_sample = '/content/faces_512/indian_0001.png' # Example reference
cctv_composite = '/content/drive/MyDrive/IVP/generated_cctv_enhanced_composite.png'

if os.path.exists(cctv_sample):
    !python /content/infer_controlnet.py \
        --controlnet_path="{final_model}" \
        --sketch_path="/content/sketches_augmented/indian_0001_orig.jpg" \
        --init_image="{cctv_sample}" \
        --strength=0.65 \
        --prompt="high quality forensic photograph of an Indian suspect, sharp facial features, highly detailed, realistic skin" \
        --output_path="{cctv_composite}"
    
    if os.path.exists(cctv_composite):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(Image.open('/content/sketches_augmented/indian_0001_orig.jpg'))
        axes[0].set_title('Forensic Sketch')
        axes[0].axis('off')
        axes[1].imshow(Image.open(cctv_sample))
        axes[1].set_title('Low-Res / CCTV Prior')
        axes[1].axis('off')
        axes[2].imshow(Image.open(cctv_composite))
        axes[2].set_title('Enhanced Suspect Composite')
        axes[2].axis('off')
        plt.tight_layout()
        plt.show()